<a href="https://colab.research.google.com/github/astro-keerthana/emotion_prediction/blob/main/audio_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount Google Drive

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Install necessary libraries

In [45]:
!pip install tensorflow tensorflow-hub transformers torch \
             torchaudio librosa soundfile praat-parselmouth -q
print("Packages installed!")

Packages installed!


# Import necessary libraries

In [46]:
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import torch
import librosa
import soundfile as sf
import parselmouth
from parselmouth.praat import call
from transformers import AutoFeatureExtractor, ASTForAudioClassification
import requests, csv, io, os, warnings
from datetime import datetime
from scipy.io import wavfile

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
print("Imports done!")

Imports done!


# Load Models

In [47]:
print("Loading YAMNet...")
yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")

print("Loading AST...")
ast_extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
ast_model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
ast_model.eval()

print("Loading YAMNet class map...")
r = requests.get(
    "https://raw.githubusercontent.com/tensorflow/models/master/"
    "research/audioset/yamnet/yamnet_class_map.csv"
)
yamnet_classes = [row[2] for row in csv.reader(io.StringIO(r.text))][1:]
print("All models loaded!")

Loading YAMNet...
Loading AST...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Loading YAMNet class map...
All models loaded!


# Danger Classes

In [48]:
DANGER_ENV = [
    "Screaming", "Glass breaking", "Gunshot, gunfire",
    "Explosion", "Crash", "Alarm", "Siren",
    "Fire alarm", "Smoke detector"
]
DANGER_PHYSIO = [
    "Cough", "Sneeze", "Wheeze", "Breathing",
    "Heartbeat", "Crying, sobbing", "Groan", "Gasp"
]
ALL_DANGER = DANGER_ENV + DANGER_PHYSIO

# Clinical Thresholds (GRBAS Standard)
CLINICAL = {
    "jitter_normal_max"  : 0.0104,
    "shimmer_normal_max" : 0.0381,
    "hnr_normal_min"     : 20.0,
}

# IM72D128 Microphone

In [49]:
MIC = {
    "sensitivity_dbfs" : -36.0,   # Sensitivity @ 94 dB SPL  → -36 dBFS ±1dB
    "aop_db"           : 128.0,   # Acoustic Overload Point   → 128 dB SPL
    "noise_floor_db"   : 22.0,    # Noise floor = 94 - SNR    → 94 - 72 = 22 dB
    "snr_db"           : 72.0,    # Signal-to-Noise Ratio     → 72 dB(A)
    "sample_rate"      : 16000,   # Supported: 16–48 kHz, we use 16 kHz
}

# Load Audio

In [50]:
def load_audio(audio_path, sr=MIC["sample_rate"]):
    audio, sr = librosa.load(audio_path, sr=sr, mono=True)
    audio     = audio / (np.max(np.abs(audio)) + 1e-9)
    return audio, sr

# Parameter 1 - dB Level

In [51]:
def get_db_level(audio_path):
    audio, sr = load_audio(audio_path)
    audio     = audio - np.mean(audio)
    rms       = np.sqrt(np.mean(audio ** 2))

    if rms < 1e-10:
        return None

    dbfs   = 20 * np.log10(rms)                 # Convert to dBFS
    db_spl = dbfs - MIC["sensitivity_dbfs"] + 94.0  # Calibrate to dB SPL

    # Soft clamp only at physical hardware limits (not arbitrary numbers)
    db_spl = float(np.clip(db_spl,
                            MIC["noise_floor_db"],
                            MIC["aop_db"]))
    return round(db_spl, 1)

# Test

In [52]:
TEST_FILE = "/content/drive/Shareddrives/wualt/project-wualt/2-output/sample-audio-data/03-01-05-01-01-01-01.wav"
print(f"\n[1] dB Level       : {get_db_level(TEST_FILE)} dB SPL")


[1] dB Level       : 127.0 dB SPL


# Parameter 2 - TWA Dose Pct

In [53]:
def get_twa_dose(audio_path):
    db_spl = get_db_level(audio_path)

    if db_spl is None:
        return 0.0

    if db_spl >= 90:
        permitted_hours = 8 / (2 ** ((db_spl - 90) / 5))
        twa             = round(min(999.9, (1 / permitted_hours) * 100), 1)
    else:
        twa = round((db_spl / 90.0) * 100 * 0.5, 1)
    return twa

# Test

In [54]:
print(f"[2] TWA Dose       : {get_twa_dose(TEST_FILE)} %")

[2] TWA Dose       : 999.9 %


# Parameter 3 - Noise

In [55]:
def get_noise_status(audio_path):
    db = get_db_level(audio_path)
    if db is None:
        return "No Signal"
    if db < 70:
        return "Safe"
    elif db < 85:
        return "Moderate"
    else:
        return "Dangerous"

# Test

In [56]:
print(f"[3] Noise Status   : {get_noise_status(TEST_FILE)}")

[3] Noise Status   : Dangerous


# Noise Score

In [57]:
def get_noise_score(audio_path):
    db = get_db_level(audio_path)
    if db is None:
        return 0.0

    # Scale across the dynamic range of IM72D128
    dynamic_range = MIC["aop_db"] - MIC["noise_floor_db"]
    score = (db - MIC["noise_floor_db"]) / dynamic_range
    return round(min(1.0, max(0.0, score)), 4)

# Test

In [58]:
print(f"[4] Noise Score    : {get_noise_score(TEST_FILE)}")

[4] Noise Score    : 0.9906


# Parameter 5 - Danger Score

In [59]:
def get_danger_score(audio_path, threshold=0.25):
    audio, sr    = load_audio(audio_path)
    audio_tf     = tf.constant(audio.astype(np.float32))
    scores, _, _ = yamnet_model(audio_tf)
    mean_scores  = tf.reduce_mean(scores, axis=0).numpy()

    yamnet_hits, yamnet_sum = {}, 0.0
    for cls in ALL_DANGER:
        for i, yc in enumerate(yamnet_classes):
            if cls.lower() in yc.lower():
                v = float(mean_scores[i])
                yamnet_hits[cls] = v
                yamnet_sum += v
                break

    inputs = ast_extractor(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = ast_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1)[0]
    id2label = ast_model.config.id2label

    ast_hits, ast_sum = {}, 0.0
    for cls in ALL_DANGER:
        for idx, label in id2label.items():
            if cls.lower() in label.lower():
                v = float(probs[idx])
                ast_hits[cls] = v
                ast_sum += v
                break

    total  = yamnet_sum + ast_sum + 1e-9
    wy, wa = yamnet_sum / total, ast_sum / total

    fused = {
        cls: round(yamnet_hits.get(cls, 0) * wy + ast_hits.get(cls, 0) * wa, 4)
        for cls in ALL_DANGER
    }
    return round(max(fused.values()), 4)

# Test

In [60]:
print(f"[5] Danger Score   : {get_danger_score(TEST_FILE)}")

[5] Danger Score   : 0.0


# Environment Score

In [61]:
def get_env_score(audio_path):
    audio, sr    = load_audio(audio_path)
    audio_tf     = tf.constant(audio.astype(np.float32))
    scores, _, _ = yamnet_model(audio_tf)
    mean_scores  = tf.reduce_mean(scores, axis=0).numpy()

    yamnet_hits, yamnet_sum = {}, 0.0
    for cls in ALL_DANGER:
        for i, yc in enumerate(yamnet_classes):
            if cls.lower() in yc.lower():
                v = float(mean_scores[i])
                yamnet_hits[cls] = v
                yamnet_sum += v
                break

    inputs = ast_extractor(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = ast_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1)[0]
    id2label = ast_model.config.id2label

    ast_hits, ast_sum = {}, 0.0
    for cls in ALL_DANGER:
        for idx, label in id2label.items():
            if cls.lower() in label.lower():
                v = float(probs[idx])
                ast_hits[cls] = v
                ast_sum += v
                break

    total  = yamnet_sum + ast_sum + 1e-9
    wy, wa = yamnet_sum / total, ast_sum / total

    fused = {
        cls: round(yamnet_hits.get(cls, 0) * wy + ast_hits.get(cls, 0) * wa, 4)
        for cls in ALL_DANGER
    }
    return round(max((fused.get(c, 0) for c in DANGER_ENV), default=0.0), 4)

print(f"[6] Env Score      : {get_env_score(TEST_FILE)}")

[6] Env Score      : 0.0


# Parameter 7 - Physio Score

In [62]:
def get_physio_score(audio_path):
    audio, sr    = load_audio(audio_path)
    audio_tf     = tf.constant(audio.astype(np.float32))
    scores, _, _ = yamnet_model(audio_tf)
    mean_scores  = tf.reduce_mean(scores, axis=0).numpy()

    yamnet_hits, yamnet_sum = {}, 0.0
    for cls in ALL_DANGER:
        for i, yc in enumerate(yamnet_classes):
            if cls.lower() in yc.lower():
                v = float(mean_scores[i])
                yamnet_hits[cls] = v
                yamnet_sum += v
                break

    inputs = ast_extractor(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = ast_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1)[0]
    id2label = ast_model.config.id2label

    ast_hits, ast_sum = {}, 0.0
    for cls in ALL_DANGER:
        for idx, label in id2label.items():
            if cls.lower() in label.lower():
                v = float(probs[idx])
                ast_hits[cls] = v
                ast_sum += v
                break

    total  = yamnet_sum + ast_sum + 1e-9
    wy, wa = yamnet_sum / total, ast_sum / total

    fused = {
        cls: round(yamnet_hits.get(cls, 0) * wy + ast_hits.get(cls, 0) * wa, 4)
        for cls in ALL_DANGER
    }
    return round(max((fused.get(c, 0) for c in DANGER_PHYSIO), default=0.0), 4)

# Test

In [63]:
print(f"[7] Physio Score   : {get_physio_score(TEST_FILE)}")

[7] Physio Score   : 0.0


# Parameter 8 - Environment Status

In [64]:
def get_env_status(audio_path, threshold=0.25):
    score = get_danger_score(audio_path, threshold)
    return "UNSAFE" if score > threshold else "SAFE"

# Test

In [65]:
print(f"[8] Env Status     : {get_env_status(TEST_FILE)}")

[8] Env Status     : SAFE


# Parameter 9 - Detection

In [66]:
def get_top_detections(audio_path, top_n=5):
    audio, sr    = load_audio(audio_path)
    audio_tf     = tf.constant(audio.astype(np.float32))
    scores, _, _ = yamnet_model(audio_tf)
    mean_scores  = tf.reduce_mean(scores, axis=0).numpy()

    yamnet_hits, yamnet_sum = {}, 0.0
    for cls in ALL_DANGER:
        for i, yc in enumerate(yamnet_classes):
            if cls.lower() in yc.lower():
                v = float(mean_scores[i])
                yamnet_hits[cls] = v
                yamnet_sum += v
                break

    inputs = ast_extractor(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = ast_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1)[0]
    id2label = ast_model.config.id2label

    ast_hits, ast_sum = {}, 0.0
    for cls in ALL_DANGER:
        for idx, label in id2label.items():
            if cls.lower() in label.lower():
                v = float(probs[idx])
                ast_hits[cls] = v
                ast_sum += v
                break

    total  = yamnet_sum + ast_sum + 1e-9
    wy, wa = yamnet_sum / total, ast_sum / total

    fused = {
        cls: round(yamnet_hits.get(cls, 0) * wy + ast_hits.get(cls, 0) * wa, 4)
        for cls in ALL_DANGER
    }
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_n]

# Test

In [67]:
print(f"[9] Top Detections :")
for cls, sc in get_top_detections(TEST_FILE):
    print(f"      {cls:<28} {sc:.4f}")

[9] Top Detections :
      Screaming                    0.0000
      Glass breaking               0.0000
      Gunshot, gunfire             0.0000
      Explosion                    0.0000
      Crash                        0.0000


# Parameter 10 - YAMNet/AST Weights

In [68]:
def get_yamnet_weight(audio_path):
    audio, sr    = load_audio(audio_path)
    audio_tf     = tf.constant(audio.astype(np.float32))
    scores, _, _ = yamnet_model(audio_tf)
    mean_scores  = tf.reduce_mean(scores, axis=0).numpy()

    yamnet_sum = sum(
        float(mean_scores[i])
        for cls in ALL_DANGER
        for i, yc in enumerate(yamnet_classes)
        if cls.lower() in yc.lower()
    )
    inputs = ast_extractor(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = ast_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1)[0]
    id2label = ast_model.config.id2label

    ast_sum = sum(
        float(probs[idx])
        for cls in ALL_DANGER
        for idx, label in id2label.items()
        if cls.lower() in label.lower()
    )
    total = yamnet_sum + ast_sum + 1e-9
    return round(yamnet_sum / total, 3)

def get_ast_weight(audio_path):
    return round(1.0 - get_yamnet_weight(audio_path), 3)

print(f"[10] YAMNet Weight : {get_yamnet_weight(TEST_FILE)}")
print(f"[11] AST Weight    : {get_ast_weight(TEST_FILE)}")

[10] YAMNet Weight : 1.0
[11] AST Weight    : 0.0


# Parameter 11 - Vocal

In [69]:
def get_pitch_hz(audio_path):
    try:
        snd        = parselmouth.Sound(audio_path)
        pitch      = call(snd, "To Pitch", 0.0, 75, 600)
        pitch_mean = call(pitch, "Get mean", 0, 0, "Hertz")
        return round(pitch_mean, 1)
    except Exception:
        return 0.0

def get_pitch_sd(audio_path):
    try:
        snd      = parselmouth.Sound(audio_path)
        pitch    = call(snd, "To Pitch", 0.0, 75, 600)
        pitch_sd = call(pitch, "Get standard deviation", 0, 0, "Hertz")
        return round(pitch_sd, 1)
    except Exception:
        return 0.0

def get_jitter(audio_path):
    try:
        snd        = parselmouth.Sound(audio_path)
        point_proc = call(snd, "To PointProcess (periodic, cc)", 75, 600)
        jitter     = call(point_proc, "Get jitter (local)",
                          0, 0, 0.0001, 0.02, 1.3)
        return round(jitter * 100, 3)
    except Exception:
        return 0.0

def get_shimmer(audio_path):
    try:
        snd        = parselmouth.Sound(audio_path)
        point_proc = call(snd, "To PointProcess (periodic, cc)", 75, 600)
        shimmer    = call([snd, point_proc], "Get shimmer (local)",
                          0, 0, 0.0001, 0.02, 1.3, 1.6)
        return round(shimmer * 100, 3)
    except Exception:
        return 0.0

def get_hnr(audio_path):
    try:
        snd         = parselmouth.Sound(audio_path)
        harmonicity = call(snd, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr         = call(harmonicity, "Get mean", 0, 0)
        return round(hnr, 2)
    except Exception:
        return 0.0

# Test

In [70]:
print(f"[12] Pitch Hz      : {get_pitch_hz(TEST_FILE)} Hz")
print(f"[13] Pitch SD      : {get_pitch_sd(TEST_FILE)} Hz")
print(f"[14] Jitter        : {get_jitter(TEST_FILE)} %")
print(f"[15] Shimmer       : {get_shimmer(TEST_FILE)} %")
print(f"[16] HNR           : {get_hnr(TEST_FILE)} dB")

[12] Pitch Hz      : 440.0 Hz
[13] Pitch SD      : 0.0 Hz
[14] Jitter        : 0.001 %
[15] Shimmer       : 0.0 %
[16] HNR           : 74.46 dB


# Stress Score

In [71]:
def get_stress_score(audio_path):
    try:
        snd         = parselmouth.Sound(audio_path)
        pitch       = call(snd, "To Pitch", 0.0, 75, 600)
        pitch_mean  = call(pitch, "Get mean", 0, 0, "Hertz")
        pitch_sd    = call(pitch, "Get standard deviation", 0, 0, "Hertz")
        point_proc  = call(snd, "To PointProcess (periodic, cc)", 75, 600)
        jitter      = call(point_proc, "Get jitter (local)",
                           0, 0, 0.0001, 0.02, 1.3)
        shimmer     = call([snd, point_proc], "Get shimmer (local)",
                           0, 0, 0.0001, 0.02, 1.3, 1.6)
        harmonicity = call(snd, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr         = call(harmonicity, "Get mean", 0, 0)

        jitter_n  = min(1.0, jitter  / CLINICAL["jitter_normal_max"])
        shimmer_n = min(1.0, shimmer / CLINICAL["shimmer_normal_max"])
        hnr_n     = min(1.0, max(0.0,
                        (CLINICAL["hnr_normal_min"] - hnr)
                        / CLINICAL["hnr_normal_min"]))
        pitch_var = min(1.0, (pitch_sd / (pitch_mean + 1e-9)) * 3)

        return round(
            (jitter_n  * 0.30) +
            (shimmer_n * 0.25) +
            (hnr_n     * 0.35) +
            (pitch_var * 0.10), 4
        )
    except Exception:
        return 0.0

def get_stress_label(audio_path):
    score = get_stress_score(audio_path)
    if score > 0.60:
        return "High Stress"
    elif score > 0.35:
        return "Moderate Stress"
    else:
        return "Calm"

# Test

In [72]:
print(f"Stress Score  : {get_stress_score(TEST_FILE)}")
print(f"Stress Label  : {get_stress_label(TEST_FILE)}")

Stress Score  : 0.0002
Stress Label  : Calm


# Verdict

In [73]:
def get_audio_score(audio_path, threshold=0.25):
    danger = get_danger_score(audio_path, threshold)
    stress = get_stress_score(audio_path)
    if danger > 0.5:
        return round((danger * 0.70) + (stress * 0.30), 4)
    else:
        return round((danger * 0.55) + (stress * 0.45), 4)

def get_verdict(audio_path):
    score = get_audio_score(audio_path)
    if score > 0.65:
        return "HIGH RISK — ALERT NOW"
    elif score > 0.35:
        return "MODERATE — MONITOR"
    else:
        return "SAFE"

# Test

In [74]:
print(f"Audio Score   : {get_audio_score(TEST_FILE)}")
print(f"Verdict       : {get_verdict(TEST_FILE)}")

Audio Score   : 0.0001
Verdict       : SAFE


# All Parameter Summary

In [78]:
def get_all_parameters(audio_path):
    print(f"  ALL PARAMETERS — {audio_path}")
    print(f"  Mic: IM72D128 | Sensitivity: {MIC['sensitivity_dbfs']} dBFS")
    print(f"  Range: {MIC['noise_floor_db']} dB → {MIC['aop_db']} dB SPL")

    params = {
        "dB_Level"       : get_db_level(audio_path),
        "TWA_Dose_pct"   : get_twa_dose(audio_path),
        "Noise_Status"   : get_noise_status(audio_path),
        "Noise_Score"    : get_noise_score(audio_path),
        "Danger_Score"   : get_danger_score(audio_path),
        "Env_Score"      : get_env_score(audio_path),
        "Physio_Score"   : get_physio_score(audio_path),
        "Env_Status"     : get_env_status(audio_path),
        "Top_Detections" : get_top_detections(audio_path),
        "YAMNet_Weight"  : get_yamnet_weight(audio_path),
        "AST_Weight"     : get_ast_weight(audio_path),
        "Pitch_Hz"       : get_pitch_hz(audio_path),
        "Pitch_SD_Hz"    : get_pitch_sd(audio_path),
        "Jitter_pct"     : get_jitter(audio_path),
        "Shimmer_pct"    : get_shimmer(audio_path),
        "HNR_dB"         : get_hnr(audio_path),
        "Stress_Score"   : get_stress_score(audio_path),
        "Stress_Label"   : get_stress_label(audio_path),
        "audio_score"    : get_audio_score(audio_path),
        "verdict"        : get_verdict(audio_path),
    }

    for k, v in params.items():
        if k == "Top_Detections":
            print(f"  {k:<20} :")
            for cls, sc in v:
                print(f"      {cls:<28} {sc:.4f}")
        else:
            print(f"  {k:<20} : {v}")

    return params

In [79]:
TEST_FILE = "/content/drive/Shareddrives/wualt/project-wualt/2-output/sample-audio-data/03-01-05-01-01-01-01.wav"

# Run Every Single Parameter & Print All at the End
all_params = get_all_parameters(TEST_FILE)


  ALL PARAMETERS — /content/drive/Shareddrives/wualt/project-wualt/2-output/sample-audio-data/03-01-05-01-01-01-01.wav
  Mic: IM72D128 | Sensitivity: -36.0 dBFS
  Range: 22.0 dB → 128.0 dB SPL
  dB_Level             : 127.0
  TWA_Dose_pct         : 999.9
  Noise_Status         : Dangerous
  Noise_Score          : 0.9906
  Danger_Score         : 0.0
  Env_Score            : 0.0
  Physio_Score         : 0.0
  Env_Status           : SAFE
  Top_Detections       :
      Screaming                    0.0000
      Glass breaking               0.0000
      Gunshot, gunfire             0.0000
      Explosion                    0.0000
      Crash                        0.0000
  YAMNet_Weight        : 1.0
  AST_Weight           : 0.0
  Pitch_Hz             : 440.0
  Pitch_SD_Hz          : 0.0
  Jitter_pct           : 0.001
  Shimmer_pct          : 0.0
  HNR_dB               : 74.46
  Stress_Score         : 0.0002
  Stress_Label         : Calm
  audio_score          : 0.0001
  verdict              